# Shooting cost evaluator

> Compose a world model (`Dynamics`) and an `Objective` into the `Costable` solvers optimise.

In [ ]:
#| default_exp planning.evaluator

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's planning package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

from typing import Callable

import torch

from stable_marl.protocols import Dynamics, Objective

In [ ]:
#| export
def flat_goal_encode(model: Dynamics, info_dict: dict, goal_obs_key: str = 'pov') -> torch.Tensor:
    """
    Encode the goal for models whose latent is one tensor: the goal observation (``goal``)
    takes the place of `goal_obs_key` (each agent's ``pov`` by default), and every
    ``goal_<key>`` the place of ``<key>`` (e.g. ``goal_position`` -> ``position``).
    Pair with :class:`GoalMSE`.
    """
    assert 'goal' in info_dict, "goal not in info_dict"
    goal = {k: v[:, 0] for k, v in info_dict.items() if torch.is_tensor(v)}   # drop the candidate axis
    goal[goal_obs_key] = goal['goal']
    for k in list(goal):
        if k.startswith('goal_'):
            goal[k[len('goal_'):]] = goal.pop(k)
    goal.pop('action', None)
    goal.pop('action_history', None)   # past actions are context, not goal
    return model.encode(goal)['emb']


def default_goal_encode(model: Dynamics, info_dict: dict) -> torch.Tensor:
    "The goal encoding used by default: :func:`flat_goal_encode` with each agent's ``pov``."
    return flat_goal_encode(model, info_dict)

In [ ]:
#| export
class ShootingCostEvaluator(torch.nn.Module):
    """
    Single-shooting adapter making ``(model, objective)`` a :class:`Costable`: it encodes the
    goal (once), rolls the candidate action sequences out with the model in one shot, then scores
    the predicted trajectories with the objective. Solvers consume it like a world model.

    Parameters
    ----------
    model : Dynamics
        World model with ``encode`` / ``rollout`` (registered as a submodule)
    objective : Objective
        Cost of a rolled-out info dict: ``(B, S)`` joint or ``(B, S, num_agents)`` per agent
    constraints : list of Objective, optional
        Terms used as constraints (satisfied when ``<= 0``), exposed by ``get_constraints``
    encode_goal : callable or None
        ``fn(model, info_dict) -> goal_emb`` (None: the caller provides ``goal_emb``)
    """
    def __init__(
        self,
        model: Dynamics,
        objective: Objective,
        constraints: list[Objective] | None = None,
        encode_goal: Callable[[Dynamics, dict], torch.Tensor] | None = default_goal_encode):
        super().__init__()
        self.model, self.objective, self.constraints, self.encode_goal = model, objective, constraints, encode_goal
        if constraints:
            self.get_constraints = self._get_constraints

    def _rollout(self, info_dict: dict, action_candidates: torch.Tensor) -> dict:
        if self.encode_goal is not None and 'goal_emb' not in info_dict:
            info_dict['goal_emb'] = self.encode_goal(self.model, info_dict)
        info_dict = self.model.rollout(info_dict, action_candidates)
        info_dict['action_candidates'] = action_candidates
        return info_dict

    def criterion(self, info_dict: dict, action_candidates: torch.Tensor | None = None) -> torch.Tensor:
        "Score an already rolled-out info dict."
        if action_candidates is not None:
            info_dict['action_candidates'] = action_candidates
        return self.objective(info_dict)

    def get_cost(self, info_dict: dict, action_candidates: torch.Tensor) -> torch.Tensor:
        "Encode the goal (if needed), roll the candidates out, then score them."
        return self.objective(self._rollout(info_dict, action_candidates))

    def _get_constraints(self, info_dict: dict, action_candidates: torch.Tensor) -> torch.Tensor:
        info_dict = self._rollout(info_dict, action_candidates)
        return torch.stack([term(info_dict) for term in self.constraints], dim=-1)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()